# Dynamic NeRF Exploration

This notebook demonstrates the core components of the Dynamic Neural Radiance Fields (Dynamic NeRF) project:
1. Setting up the environment
2. Initializing the positional encoders and model
3. Running a quick forward pass
4. Verifying the dataset loader (requires preprocessed data)
5. A mini training loop on synthetic random data

## 1. Setup

In [ ]:
import sys
import os

# Add the src directory to the Python path so we can import project modules
src_dir = os.path.abspath(os.path.join('..', 'src'))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Positional Encoder

In [ ]:
from models.nerf import PositionalEncoder

# Position encoder: 3D xyz → 63-dimensional encoding (3 * (1 + 2*10))
pos_encoder = PositionalEncoder(input_dims=3, num_freqs=10).to(device)
print(f'Position encoder output dims: {pos_encoder.num_output_dims}')  # expected: 63

# Direction encoder: 3D dir → 27-dimensional encoding (3 * (1 + 2*4))
dir_encoder = PositionalEncoder(input_dims=3, num_freqs=4).to(device)
print(f'Direction encoder output dims: {dir_encoder.num_output_dims}')  # expected: 27

# Test forward pass
x = torch.rand(8, 3, device=device)
encoded = pos_encoder(x)
print(f'Input shape: {x.shape}, Encoded shape: {encoded.shape}')

## 3. NeRF Model – Forward Pass

In [ ]:
from models.nerf import NeRF

model = NeRF(hidden_dims=256).to(device)
print(model)

rays_o = torch.rand(4, 3, device=device)
rays_d = F.normalize(torch.rand(4, 3, device=device), dim=-1)

rgb, density = model(rays_o, rays_d)
print(f'RGB shape: {rgb.shape}, Density shape: {density.shape}')

## 4. Dynamic NeRF Model – Forward Pass

In [ ]:
from models.dynamic_nerf import DynamicNeRF, TemporalEncoder

dyn_model = DynamicNeRF(hidden_dims=256, use_attention=True).to(device)
print(dyn_model)

rays_o = torch.rand(4, 3, device=device)
rays_d = F.normalize(torch.rand(4, 3, device=device), dim=-1)
time_idx = torch.rand(4, 1, device=device)  # normalised in [0, 1]

rgb, density = dyn_model(rays_o, rays_d, time_idx)
print(f'RGB shape: {rgb.shape}, Density shape: {density.shape}')

## 5. Volume Rendering

In [ ]:
rays_o = torch.rand(4, 3, device=device)
rays_d = F.normalize(torch.rand(4, 3, device=device), dim=-1)
time_idx = torch.rand(4, 1, device=device)

rgb_vol, depth, weights = dyn_model.volume_rendering(
    rays_o, rays_d, time_idx, near=2.0, far=6.0, num_samples=32, rand=False
)
print(f'Rendered RGB: {rgb_vol.shape}, Depth: {depth.shape}, Weights: {weights.shape}')

## 6. Ray Generation

In [ ]:
from utils.ray_utils import get_rays

H, W, focal = 64, 64, 50.0
# Identity camera pose (camera at world origin)
c2w = torch.eye(4)

rays_o, rays_d = get_rays(H, W, focal, c2w)
print(f'rays_o shape: {rays_o.shape}')  # (H, W, 3)
print(f'rays_d shape: {rays_d.shape}')  # (H, W, 3)

# Visualize ray direction magnitudes
norms = rays_d.norm(dim=-1).numpy()
plt.figure(figsize=(5, 4))
plt.imshow(norms, cmap='viridis')
plt.colorbar(label='Direction magnitude')
plt.title('Ray direction magnitudes')
plt.tight_layout()
plt.show()

## 7. Mini Training Loop (Synthetic Data)

Demonstrates the training pipeline on randomly generated data without requiring a real dataset.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# --- Synthetic dataset ---
N_rays = 512
rays_o_data = torch.rand(N_rays, 3)
rays_d_data = F.normalize(torch.rand(N_rays, 3), dim=-1)
time_data   = torch.rand(N_rays, 1)
target_rgb  = torch.rand(N_rays, 3)  # random "ground-truth" colours

dataset = TensorDataset(rays_o_data, rays_d_data, time_data, target_rgb)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

# --- Model & optimiser ---
mini_model = DynamicNeRF(hidden_dims=64, use_attention=False).to(device)
optimizer  = torch.optim.Adam(mini_model.parameters(), lr=5e-4)

# --- Training loop ---
losses = []
for epoch in range(1, 6):
    epoch_loss = 0.0
    for rays_o_b, rays_d_b, time_b, rgb_b in loader:
        rays_o_b = rays_o_b.to(device)
        rays_d_b = rays_d_b.to(device)
        time_b   = time_b.to(device)
        rgb_b    = rgb_b.to(device)

        optimizer.zero_grad()
        pred_rgb, _ = mini_model(rays_o_b, rays_d_b, time_b)
        loss = F.mse_loss(pred_rgb, rgb_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    losses.append(avg)
    print(f'Epoch {epoch}/5  loss={avg:.6f}')

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(losses)+1), losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Mini training curve (synthetic data)')
plt.tight_layout()
plt.show()

## 8. Dataset Loader (requires preprocessed data)

Uncomment and adjust the paths below once you have run `src/scripts/preprocess_data.py`.

In [ ]:
# from data.dataset import DynamicNeRFDataset
#
# dataset = DynamicNeRFDataset(
#     root_dir='../data',
#     scene_name='lego',
#     split='train',
#     img_size=(200, 200),
# )
#
# print(f'Dataset size: {len(dataset)} rays')
# print(f'HWF: {dataset.hwf}')
# print(f'Poses shape: {dataset.poses.shape}')
#
# rays_o, rays_d, time_idx, rgb = dataset[0]
# print(f'Sample – rays_o: {rays_o.shape}, time: {time_idx.item():.4f}, rgb: {rgb}')

## Summary

| Component | Status |
|-----------|--------|
| `PositionalEncoder` (buffer-registered `freq_bands`) | ✅ |
| `NeRF` forward pass | ✅ |
| `DynamicNeRF` forward pass | ✅ |
| Volume rendering | ✅ |
| Ray generation (`get_rays`) | ✅ |
| `DynamicNeRFDataset` | ✅ (needs data) |
| Mini training loop | ✅ |